# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset package using the `mlcroissant` library, leveraging the Croissant schema and unique `@id` references.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and inspect
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their field `@id` values using metadata. All entities are referenced by their `@id` as required by Croissant conventions.

In [ ]:
def print_recordsets_overview(ds):
    print("Available Record Sets (@id):")
    for record_set in ds.metadata.record_sets:
        print(f"- {record_set['@id']}: {record_set.get('name', 'No name')}")
        fields = record_set.get('fields', [])
        if fields:
            print("  Fields:")
            for field in fields:
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
                print(f"    - {field_id}")
        else:
            print("  No fields listed.")

print_recordsets_overview(dataset)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the above overview.

In [ ]:
# Identify all record set @ids
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

# Extract data for each record set, indexed by their @id
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded Record Set {record_set_id} with {len(df)} rows.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

if dataframes:
    first_rs = record_sets[0]
    print(f"\nColumns in record set '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()
else:
    print("No record sets with data loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, grouping, etc. on a numeric field from the selected record set. 

_Note:_ Replace `<record_set_id>`, `<numeric_field_id>`, `<group_field_id>` with the correct `@id`s as identified above.

In [ ]:
# Example: Pick the first available record set with data
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")

    # Try to select a numeric field by inspecting dtypes
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notna().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()

        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
        print(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try to find a categorical/grouping field
        group_field = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() < 10:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_' + numeric_field)
            print(f"Grouped data by {group_field} (showing mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for analysis in this record set.")
else:
    print("No data found in any record set.")

## 5. Visualization
Visualize the distribution of the chosen numeric field, and if a group field is available, plot group-wise means.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    df[numeric_field].dropna().hist(bins=20)
    plt.title(f"Distribution of {numeric_field} in record set '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping was performed above
    if 'grouped_df' in locals() and group_field and not grouped_df.empty:
        grouped_df.plot(kind='bar', legend=False, figsize=(8, 3))
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Group-wise mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
With the `mlcroissant` library, you can load FAIR-format datasets described by Croissant schemas, navigate record sets and their `@id`s, and perform exploratory data analysis directly from the schema URL. 

- All entities in this notebook were referenced by their `@id` for traceability and reproducibility.
- This approach is robust to schema changes and enforces strict and transparent data provenance.

For further analysis, consult the dataset documentation for field definitions and limitations.

_End of Notebook._